# 模型调用

## 文本模型

### 调用混元大模型

In [2]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("HUNYUAN_API_KEY")

model = ChatOpenAI(
    model="hunyuan-lite",
    base_url="https://api.hunyuan.cloud.tencent.com/v1",
    api_key=api_key
)

# 调用
# response = model.invoke("简单解释一下什么是量子纠缠")
# print(response.content)

### 调用deepseek

In [ ]:
load_dotenv()

api_key = os.getenv("DEEPSEEK_API_KEY")

model = ChatOpenAI(
    model="deepseek-v4-flash",
    base_url="https://api.deepseek.com/v1",
    api_key=api_key
)

# 调用
response = model.invoke("你是哪个模型？")
print(response.content)

api_key=sk-94b570f6c42f4e02bcfd4f2fb1379ad8
我是DeepSeek最新版本的模型，由深度求索公司开发！具体来说是DeepSeek最新版模型（可能不是DeepSeek-V3或DeepSeek-R1，而是后续的迭代版本）。我会根据上下文和技术栈不断进化，旨在提供准确、高效的智能助手服务。

有什么我可以帮助你的吗？无论是回答问题、处理文档、进行推理分析，还是日常闲聊，我都很乐意配合！😊


### 调用七牛云AI

In [3]:
load_dotenv()

api_key = os.getenv("QINIUYUN_API_KEY")
print(f"api_key={api_key}")

model = ChatOpenAI(
    model="deepseek-v3",
    base_url="https://ai.qiniuapi.com/v1",
    api_key=api_key,
    temperature=0.0
)

# 调用
response = model.invoke("你是哪个模型？")
print(response.content)


api_key=sk-9220555154a0167b8d669fcf0a29ac092df6a8e02c0bb20a8ea7080db559098a


AuthenticationError: Error code: 401 - {'status_code': 401, 'code': 401, 'message': 'bad token'}

### 调用硅基流动

In [28]:
load_dotenv()

api_key = os.getenv("SILICONFLOW_API_KEY")

model = ChatOpenAI(
    model="Pro/zai-org/GLM-4.7",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    temperature=0.0
)

# 调用
response = model.invoke("你是哪个模型？")
print(response.content)

api_key=sk-wdpmumyyzzdvwgugbyjojxplkjgnnezyxtwrxqjntqkoxkcl
我是Z.ai开发的GLM（General Language Model）大语言模型，基于Transformer架构构建。

我通过大规模文本数据训练，能够理解和生成人类语言，回答问题、提供信息和进行对话交流。我的设计目标是帮助用户解决问题、提供创意支持，并确保回答的安全性和准确性。

有什么我能帮您解答的问题吗？


## 文生图模型

### 硅基流动

In [30]:
api_key = os.getenv("SILICONFLOW_API_KEY")

model = ChatOpenAI(
    model="Qwen/Qwen-Image",
    base_url="https://api.siliconflow.cn/v1",
    api_key=api_key,
    temperature=0.0
)

# 调用
response = model.invoke("创建一张猫的图片")
print(response.content)

BadRequestError: Error code: 400 - {'code': 20012, 'message': 'Model does not exist. Please check it carefully.', 'data': None}

# 工具调用

## 并行工具调用

In [7]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """获取某个位置的天气。"""
    return f"{location} 天气晴朗。"

model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke(
    "波士顿和东京的天气怎么样？"
)

print(response.tool_calls)

# 执行所有工具（可以使用 async 并行执行）
results = []
for tool_call in response.tool_calls:
    if tool_call['name'] == 'get_weather':
        result = get_weather.invoke(tool_call)
    ...
    results.append(result)

print(results)

[{'name': 'get_weather', 'args': {'location': '波士顿'}, 'id': 'call_d7ti7fk2c3mff5situo0', 'type': 'tool_call'}, {'name': 'get_weather', 'args': {'location': '东京'}, 'id': 'call_d7ti7fk2c3mff5situog', 'type': 'tool_call'}]
[ToolMessage(content='波士顿 天气晴朗。', name='get_weather', tool_call_id='call_d7ti7fk2c3mff5situo0'), ToolMessage(content='东京 天气晴朗。', name='get_weather', tool_call_id='call_d7ti7fk2c3mff5situog')]


## 流式传输工具调用

In [8]:
for chunk in model_with_tools.stream(
    "波士顿和东京的天气怎么样？"
):
    # 工具调用块逐步到达
    for tool_chunk in chunk.tool_call_chunks:
        if name := tool_chunk.get("name"):
            print(f"工具：{name}")
        if id_ := tool_chunk.get("id"):
            print(f"ID：{id_}")
        if args := tool_chunk.get("args"):
            print(f"参数：{args}")

工具：get_weather
ID：call_d7tiauk2c3m1fv3m3lk0
ID：call_d7tiauk2c3m1fv3m3lk0
参数：{"location":"波士顿"}
工具：get_weather
ID：call_d7tiauk2c3m1fv3m3lkg
ID：call_d7tiauk2c3m1fv3m3lkg
参数：{"location":"东京"}


# 结构化输出

## Pydantic

In [25]:
from pydantic import BaseModel, Field, AliasChoices

class Movie(BaseModel):
    """一部带有详细信息的电影。"""
    # 使用 AliasChoices，模型返回 year 或 release_year 都能解析成功
    year: int = Field(
        ..., 
        validation_alias=AliasChoices('year', 'release_year'),
        description="电影上映年份"
    )
    title: str = Field(..., description="电影名称")
    director: str = Field(..., description="导演姓名")
    # 评分容易漏，给它一个默认值或者更强烈的描述
    rating: float = Field(
        default=0.0, 
        description="电影评分，如果不知道请填 0.0"
    )

model_with_structure = model.with_structured_output(Movie, method="json_mode", include_raw=True)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的详细信息。请直接输出 JSON，不要包含任何解释性文字。")
print(response)

{'raw': AIMessage(content='{\n  "title": "盗梦空间",\n  "original_title": "Inception",\n  "director": "克里斯托弗·诺兰",\n  "release_year": 2010,\n  "genre": ["科幻", "动作", "悬疑"],\n  "cast": ["莱昂纳多·迪卡普里奥", "约瑟夫·高登-莱维特", "艾伦·佩吉", "汤姆·哈迪", "渡边谦", "迪利普·劳", "玛丽昂·歌迪亚", "基里安·墨菲", "迈克尔·凯恩"],\n  "plot": "道姆·柯布（莱昂纳多·迪卡普里奥 饰）是经验老道的窃贼，专门在人们梦境之中盗取潜意识中的秘密。为了回到儿女身边并洗清罪名，他接受了一项艰巨任务：在目标人物的潜意识中植入一个想法，即“意念植入”。为此，他带领团队层层深入多重梦境，却面临来自潜意识防御的致命攻击以及自身心魔的困扰。",\n  "rating": 9.3,\n  "language": "英语",\n  "country": "美国",\n  "runtime": "148分钟"\n}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 517, 'prompt_tokens': 47, 'total_tokens': 564, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 259, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 47}, 'model_provider': 'openai', 'model_name': 'deepse

## JSON Schema

In [18]:
import json

json_schema = {
    "title": "Movie",
    "description": "一部带有详细信息的电影",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "电影标题"
        },
        "year": {
            "type": "integer",
            "description": "电影上映年份"
        },
        "director": {
            "type": "string",
            "description": "电影导演"
        },
        "rating": {
            "type": "number",
            "description": "电影评分，满分 10 分"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("提供关于电影《盗梦空间》的详细信息")
print(response)

BadRequestError: Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

## JsonOutputParser

In [19]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

# 初始化解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 创建包含格式指令的模板
prompt = PromptTemplate(
    template="回答用户查询。\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | model | parser

response = chain.invoke({"query": "提供关于电影《盗梦空间》的详细信息"})
print(response)

{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'rating': 8.8}


# 高级主题

## 回调处理程序

In [32]:
from langchain_core.callbacks import UsageMetadataCallbackHandler

callback = UsageMetadataCallbackHandler()
result_1 = model.invoke("你好", config={"callbacks": [callback]})
callback.usage_metadata

{'hunyuan-lite': {'input_tokens': 3,
  'output_tokens': 33,
  'total_tokens': 36,
  'input_token_details': {},
  'output_token_details': {}}}

### 自定义回调函数

In [33]:
from langchain_core.callbacks import BaseCallbackHandler

class MySimpleTracker(BaseCallbackHandler):
    """一个自定义的简易 Token 追踪回调"""
    
    def __init__(self):
        self.total_tokens = 0

    # 这是一个预定义好的回调函数名称，不能改名
    def on_llm_end(self, response, **kwargs):
        """当模型运行结束时，这个函数会自动被触发"""
        # 从响应中提取 token 消耗
        usage = response.llm_output.get("token_usage", {}) # 不同模型的 key 可能略有不同
        tokens = usage.get("total_tokens", 0)
        
        self.total_tokens += tokens
        print(f"检测到本次调用消耗: {tokens} tokens")

# 使用方法
tracker = MySimpleTracker()
model.invoke("你好", config={"callbacks": [tracker]})

检测到本次调用消耗: 35 tokens


AIMessage(content='你好！很高兴与你交流。请问有什么我可以帮助你的吗？无论是关于生活、工作、学习还是其他方面的问题，我都会尽力回答你。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 3, 'total_tokens': 35, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'hunyuan-lite', 'system_fingerprint': '', 'id': 'defb7781ac12a8cac61982ad844a5763', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dfd99-8fe2-7340-8277-e620d38d4f55-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 32, 'total_tokens': 35, 'input_token_details': {}, 'output_token_details': {}})